In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 45.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 79.4 MB/s eta 0:00:00:00:01:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.3 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=fbfd260a0d7249e7ac5ac9aaf63fb2eab949ac3c67d625558fecee7940b517b4
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import math

backend = AerSimulator()

In [3]:
# === SHARED UTILITY ===
# Quantum random bit generator. The protocol requires several random choices (Alice's bits, Alice's bases, Bob's bases, sample selection
# for the eavesdropping check, and Eve's bases in the attacker version).
# We obtain randomness by measuring |+> in the computational basis:
# applying H to |0> gives (|0> + |1>)/sqrt(2), and measuring in Z
# yields 0 or 1 with equal probability.

# To amortise the simulator cost we build one k-qubit Hadamard circuit
# at a time and read k independent random bits from a single shot.
# We chunk to keep individual circuits small.

_CHUNK = 24  # bits per circuit

def _random_chunk(k):
    qc = QuantumCircuit(k, k)
    qc.h(range(k))
    qc.measure(range(k), range(k))
    counts = backend.run(qc, shots=1).result().get_counts()
    bitstring = next(iter(counts))
    # Qiskit prints the highest-index qubit on the left; reverse so
    # index i in the returned list corresponds to qubit i.
    return [int(b) for b in reversed(bitstring)]

def quantum_random_bits(n):
    """Return a list of n random bits from measuring |+> states in Z."""
    bits = []
    remaining = n
    while remaining > 0:
        k = min(_CHUNK, remaining)
        bits.extend(_random_chunk(k))
        remaining -= k
    return bits

def quantum_random_bit():
    return quantum_random_bits(1)[0]

# Sanity check: should be roughly half 0s and half 1s.
sample = quantum_random_bits(1000)
print(f"Sample of 1000 quantum random bits: {sum(sample)} ones, {1000 - sum(sample)} zeros")

Sample of 1000 quantum random bits: 515 ones, 485 zeros


In [4]:
# ALICE'S CODE 
# For each transmission Alice picks a random bit and a random basis:
#   basis 0 = Z (computational): bit 0 -> |0>,  bit 1 -> |1>
#   basis 1 = X (diagonal):      bit 0 -> |+>,  bit 1 -> |->
# She returns a 1-qubit circuit that prepares that state.

def alice_prepare(bit, basis):
    """Return a 1-qubit circuit preparing Alice's chosen state."""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)
    if basis == 1:
        qc.h(0)
    return qc

In [5]:
# BOB'S CODE
# To measure in basis b, Bob applies H if b = 1 (rotating the X-basis
# back to the computational basis) and then measures in Z. If his
# basis matches Alice's, his result equals Alice's bit; otherwise the
# outcome is uniformly random.

def bob_measure(qc, basis):
    """Append Bob's measurement to qc, run it, return his classical bit."""
    if basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    counts = backend.run(qc, shots=1).result().get_counts()
    bitstring = next(iter(counts))
    return int(bitstring[0])

In [6]:
# EVE'S CODE

def eve_intercept_resend(qc_from_alice, eve_basis):
    """Measure Alice's qubit in eve_basis, then prepare a fresh
    qubit encoding the measured bit in eve_basis to forward to Bob.
    Returns (new_circuit, eve_bit).
    """
    # --- Eve's measurement on Alice's qubit ---
    if eve_basis == 1:
        qc_from_alice.h(0)
    qc_from_alice.measure(0, 0)
    counts = backend.run(qc_from_alice, shots=1).result().get_counts()
    eve_bit = int(next(iter(counts))[0])

    # --- Eve re-prepares a qubit and forwards it ---
    qc_new = QuantumCircuit(1, 1)
    if eve_bit == 1:
        qc_new.x(0)
    if eve_basis == 1:
        qc_new.h(0)
    return qc_new, eve_bit

In [7]:
# SHARED HELPER
# Sifting step: Alice and Bob publicly announce their bases (but not
# their bits) and keep only the positions where the bases agree.

def sift(alice_bits, alice_bases, bob_bits, bob_bases):
    matching = [i for i in range(len(alice_bases)) if alice_bases[i] == bob_bases[i]]
    a_sift = [alice_bits[i] for i in matching]
    b_sift = [bob_bits[i]   for i in matching]
    return a_sift, b_sift, matching

In [8]:
def transmit_qubit_with_eve(alice_bit, alice_basis, eve_basis, bob_basis, attack=True):
    """Single Alice -> (Eve) -> Bob round.
    Returns (bob_bit, eve_bit_or_None)."""
    qc = alice_prepare(alice_bit, alice_basis)            # ALICE
    if attack:
        qc, eve_bit = eve_intercept_resend(qc, eve_basis) # EVE
    else:
        eve_bit = None
    bob_bit = bob_measure(qc, bob_basis)                  # BOB
    return bob_bit, eve_bit

In [9]:
# BASELINE: NO ATTACKER
# Run the full protocol WITHOUT Eve to confirm:
# a) sifted keys agree perfectly on a noiseless channel
# b) the eavesdropping-check threshold raises no false alarm

N_BASELINE = 400
THRESHOLD = 0.15

a_bits = quantum_random_bits(N_BASELINE)
a_bases = quantum_random_bits(N_BASELINE)
b_bases = quantum_random_bits(N_BASELINE)
e_bases = quantum_random_bits(N_BASELINE)  # allocated but unused (attack=False)

b_results_no_eve = []
for i in range(N_BASELINE):
    bb, _ = transmit_qubit_with_eve(
        a_bits[i], a_bases[i], e_bases[i], b_bases[i], attack=False #Eve does NOT intercept
    )
    b_results_no_eve.append(bb)

sa_ne, sb_ne, _ = sift(a_bits, a_bases, b_results_no_eve, b_bases)

# Eavesdropping check using a sacrificed quantum-random sample
sample_mask_ne = quantum_random_bits(len(sa_ne))
sample_idx_ne = [k for k, m in enumerate(sample_mask_ne) if m == 1]
remain_idx_ne = [k for k, m in enumerate(sample_mask_ne) if m == 0]

n_cmp_ne = len(sample_idx_ne)
n_dis_ne = sum(1 for k in sample_idx_ne if sa_ne[k] != sb_ne[k])
err_ne = n_dis_ne / n_cmp_ne if n_cmp_ne else 0.0

print('BASELINE: NO ATTACKER')
print(f'Qubits sent:              {N_BASELINE}')
print(f'Sifted key length:        {len(sa_ne)}')
print(f'Sample size (sacrificed): {n_cmp_ne}')
print(f'Disagreements in sample:  {n_dis_ne}')
print(f'Observed error rate:      {err_ne:.3f}')
print(f'Threshold:                {THRESHOLD}')

if err_ne > THRESHOLD:
    print('\n*** FALSE ALARM *** -- should NOT happen on a noiseless channel.')
    assert False, 'No-attack run should never trigger the threshold.'
else:
    final_alice_ne = [sa_ne[k] for k in remain_idx_ne]
    final_bob_ne   = [sb_ne[k] for k in remain_idx_ne]
    assert final_alice_ne == final_bob_ne, 'Keys must agree with no Eve and no noise.'
    print(f'\nNo attack detected (correct). Final key length: {len(final_alice_ne)}')
    print('Keys agree perfectly -- no false alarm. ✓')


BASELINE: NO ATTACKER
Qubits sent:              400
Sifted key length:        222
Sample size (sacrificed): 106
Disagreements in sample:  0
Observed error rate:      0.000
Threshold:                0.15

No attack detected (correct). Final key length: 116
Keys agree perfectly -- no false alarm. ✓


In [10]:
N = 400

alice_bits  = quantum_random_bits(N)
alice_bases = quantum_random_bits(N)
eve_bases   = quantum_random_bits(N)
bob_bases   = quantum_random_bits(N)

bob_results = []
eve_bits    = []
for i in range(N):
    bb, eb = transmit_qubit_with_eve(
        alice_bits[i], alice_bases[i],
        eve_bases[i],  bob_bases[i],
        attack=True,
    )
    bob_results.append(bb)
    eve_bits.append(eb)

sifted_alice, sifted_bob, matching = sift(alice_bits, alice_bases, bob_results, bob_bases)
print(f"Qubits sent:               {N}")
print(f"Sifted key length:         {len(sifted_alice)}")

Qubits sent:               400
Sifted key length:         200


In [11]:
THRESHOLD = 0.15

# Random sample mask over the sifted positions.
sample_mask    = quantum_random_bits(len(sifted_alice))
sample_idx     = [k for k, m in enumerate(sample_mask) if m == 1]
remaining_idx  = [k for k, m in enumerate(sample_mask) if m == 0]

n_compared = len(sample_idx)
n_disagree = sum(1 for k in sample_idx if sifted_alice[k] != sifted_bob[k])
error_rate = n_disagree / n_compared if n_compared else 0.0

print(f"Sample size (sacrificed):   {n_compared}")
print(f"Disagreements in sample:    {n_disagree}")
print(f"Observed error rate:        {error_rate:.3f}")
print(f"Threshold:                  {THRESHOLD}")

if error_rate > THRESHOLD:
    print("\n*** ATTACK DETECTED *** -- aborting; no key is generated.")
else:
    final_alice = [sifted_alice[k] for k in remaining_idx]
    final_bob   = [sifted_bob[k]   for k in remaining_idx]
    print(f"\nNo attack signal. Final key length: {len(final_alice)}")
    print(f"Final key (Alice): {''.join(str(b) for b in final_alice)}")
    print(f"Final key (Bob):   {''.join(str(b) for b in final_bob)}")

Sample size (sacrificed):   87
Disagreements in sample:    26
Observed error rate:        0.299
Threshold:                  0.15

*** ATTACK DETECTED *** -- aborting; no key is generated.


In [12]:
def run_one_trial(N, attack_prob):
    """Run the full protocol once. Each qubit is attacked
    independently with probability attack_prob (using quantum-random
    bits to decide). Returns the observed sifted error rate."""
    a_bits  = quantum_random_bits(N)
    a_bases = quantum_random_bits(N)
    e_bases = quantum_random_bits(N)
    b_bases = quantum_random_bits(N)

    # Decide which qubits Eve attacks. We use the fact that a
    # sum of k independent quantum-random bits is in [0,k]; to
    # get a Bernoulli(p) with p = a/2^k, we threshold the integer.
    # For a flexible probability we threshold a random integer
    # drawn from enough random bits.
    PRECISION_BITS = 10
    threshold_int  = int(attack_prob * (1 << PRECISION_BITS))
    attack_flags   = []
    for _ in range(N):
        bs = quantum_random_bits(PRECISION_BITS)
        val = sum(b << j for j, b in enumerate(bs))
        attack_flags.append(val < threshold_int)

    b_bits = []
    for i in range(N):
        bb, _ = transmit_qubit_with_eve(
            a_bits[i], a_bases[i],
            e_bases[i], b_bases[i],
            attack=attack_flags[i],
        )
        b_bits.append(bb)

    sa, sb, _ = sift(a_bits, a_bases, b_bits, b_bases)
    # The eavesdropping-check sample (here just the whole sifted key
    # for the cleanest error-rate estimate).
    if len(sa) == 0:
        return 0.0, 0
    disagree = sum(1 for a, b in zip(sa, sb) if a != b)
    return disagree / len(sa), len(sa)

TRIALS = 5
N      = 300
print(f"{'trial':>5}  {'sifted_len':>10}  {'error_rate':>10}  {'detected (> {:.2f})':>20}".format(0.15))
for t in range(TRIALS):
    rate, slen = run_one_trial(N, attack_prob=1.0)
    print(f"{t:>5}  {slen:>10}  {rate:>10.3f}  {str(rate > 0.15):>20}")

trial  sifted_len  error_rate   detected (> 0.15)
    0         157       0.255                  True
    1         143       0.287                  True
    2         161       0.230                  True
    3         139       0.230                  True
    4         141       0.248                  True


In [13]:
# MULTI-TRIAL HELPER (with proper sacrificed sample)
# run_one_trial above uses the ENTIRE sifted key for the error
# check, leaving nothing as a final key. run_one_trial_v2 instead
# sacrifices a quantum-random half-sample for the check and keeps
# the rest as the usable key, matching the protocol in the cell above.

def run_one_trial_v2(N, attack_prob):
    """Run one full BB84 round with optional partial Eve attack.
    Returns (error_rate_on_sample, sifted_len, final_key_len).
    """
    a_bits  = quantum_random_bits(N)
    a_bases = quantum_random_bits(N)
    e_bases = quantum_random_bits(N)
    b_bases = quantum_random_bits(N)

    PRECISION_BITS = 10
    threshold_int  = int(attack_prob * (1 << PRECISION_BITS))
    attack_flags   = []
    for _ in range(N):
        bs  = quantum_random_bits(PRECISION_BITS)
        val = sum(b << j for j, b in enumerate(bs))
        attack_flags.append(val < threshold_int)

    b_bits = []
    for i in range(N):
        bb, _ = transmit_qubit_with_eve(
            a_bits[i], a_bases[i],
            e_bases[i], b_bases[i],
            attack=attack_flags[i],
        )
        b_bits.append(bb)

    sa, sb, _ = sift(a_bits, a_bases, b_bits, b_bases)
    if len(sa) == 0:
        return 0.0, 0, 0

    # Sacrifice a quantum-random sample for the eavesdropping check
    sample_mask = quantum_random_bits(len(sa))
    sample_idx  = [k for k, m in enumerate(sample_mask) if m == 1]
    remain_idx  = [k for k, m in enumerate(sample_mask) if m == 0]

    n_cmp    = len(sample_idx)
    n_dis    = sum(1 for k in sample_idx if sa[k] != sb[k])
    err_rate = n_dis / n_cmp if n_cmp else 0.0
    key_len  = len(remain_idx)

    return err_rate, len(sa), key_len


TRIALS = 5
N      = 300
print(f"{'trial':>5}  {'sifted_len':>10}  {'key_len':>8}  "
      f"{'error_rate':>10}  {'detected (>0.15)':>18}")
for t in range(TRIALS):
    rate, slen, klen = run_one_trial_v2(N, attack_prob=1.0)
    print(f"{t:>5}  {slen:>10}  {klen:>8}  {rate:>10.3f}  {str(rate > 0.15):>18}")


trial  sifted_len   key_len  error_rate    detected (>0.15)
    0         148        68       0.325                True
    1         151        76       0.307                True
    2         160        71       0.270                True
    3         144        72       0.167                True
    4         144        71       0.247                True


In [14]:
# ATTACK-PROBABILITY SWEEP
# Sweep Eve's attack probability from 0 (no attack) to 1 (full).
# Expected error rate in the sifted key ~ p/4:
#   - Eve guesses Alice's basis correctly 50% of the time.
#   - A wrong-basis measurement disturbs the qubit, causing a
#     25% error rate in Bob's matching-basis positions.
# At p=0.0 the error rate should be ~0 and no alarm fires.
# At p>=0.6 the rate reliably exceeds the 0.15 threshold.

import math

N_SWEEP = 400
print(f"{'p':>6}  {'sifted':>7}  {'key_len':>8}  "
      f"{'observed':>10}  {'expected~':>10}  {'detected':>10}")
for p in [0.0, 0.1, 0.25, 0.5, 0.75, 1.0]:
    rate, slen, klen = run_one_trial_v2(N_SWEEP, attack_prob=p)
    expected = p / 4
    detected = rate > 0.15
    print(
        f"p={p:>4.2f}  {slen:>7}  {klen:>8}  "
        f"{rate:>10.3f}  {expected:>10.3f}  {str(detected):>10}"
    )
    if p == 0.0:
        assert not detected, 'False alarm on p=0 -- something is wrong!'
        print('  ^ error rate ≈ 0, no false alarm.')
    if p == 1.0:
        assert detected, 'Full attack NOT detected -- threshold may be too high!'
        print('  ^ full attack detected as expected.')

     p   sifted   key_len    observed   expected~    detected
p=0.00      186        83       0.000       0.000       False
  ^ error rate ≈ 0, no false alarm.
p=0.10      206        94       0.000       0.025       False
p=0.25      183       101       0.061       0.062       False
p=0.50      198       111       0.138       0.125       False
p=0.75      192        88       0.173       0.188        True
p=1.00      216       114       0.265       0.250        True
  ^ full attack detected as expected.
